# Gating Risky Agent Actions with RelayShield

Agents that can connect to arbitrary MCP servers or install packages at runtime have a real supply-chain problem: the MCP ecosystem's registry security is minimal industry-wide, so a typosquat or a newly-registered malicious server can look identical to a legitimate one at the moment an agent decides to connect. A tool the model *may* call to check first isn't the same guarantee as a check the framework *enforces* before the connection happens — the model can simply skip it.

This cookbook covers two ways [RelayShield](https://relayshield.net) plugs into the [Agents SDK](https://openai.github.io/openai-agents-python/) to close that gap:

1. **Advisory tools** the agent can call directly — `check_mcp_server_risk` and `check_prompt_injection_breach`.
2. **A mandatory pre-execution gate**, built on the SDK's `@tool_input_guardrail` hook, that blocks a protected connect/install call outright on a risk finding — before the underlying tool ever runs, regardless of what the model decides.

## Requirements

- Python 3.10+
- An OpenAI API key, exported as `OPENAI_API_KEY`
- A RelayShield API key, exported as `RELAYSHIELD_API_KEY` — get one at [api.relayshield.net/developers](https://api.relayshield.net/developers)
- The `openai-agents-relayshield` package (installed below), which pulls in `openai-agents` as a dependency

## Install dependencies

In [ ]:
%pip install openai-agents-relayshield --quiet

## Imports and environment

Both RelayShield tools take `api_key` as a call argument rather than reading it from the environment implicitly — a shared agent process may act on behalf of multiple callers with different RelayShield keys, so the key is passed per call, not baked into a single shared config.

In [ ]:
import os

from agents import Agent, Runner
from openai_agents_relayshield import check_mcp_server_risk, check_prompt_injection_breach

RELAYSHIELD_API_KEY = os.environ["RELAYSHIELD_API_KEY"]

## Part 1 — advisory tools

Give the agent both checks directly. `check_mcp_server_risk` flags known-malicious IOC matches, typosquat domains close to well-known MCP ecosystem names, and newly-registered domains hosting the server — provide `server_url`, `package_name`, or both. `check_prompt_injection_breach` checks whether an email appears in RelayShield's stolen-session corpus with a suspected-agentic-source marker: a session/token exposure that shows signs of having been captured via a compromised or hijacked AI agent rather than a conventional phishing path.

In [ ]:
agent = Agent(
    name="Assistant",
    instructions="You help the user evaluate whether it's safe to connect to third-party MCP servers.",
    tools=[check_mcp_server_risk, check_prompt_injection_breach],
)

result = await Runner.run(
    agent,
    f"My RelayShield key is {RELAYSHIELD_API_KEY}. Is it safe to connect to the MCP server at https://mcp.example.com/sse?",
)
print(result.final_output)

An advisory tool only helps if the model actually calls it. Nothing above stops the agent from connecting anyway if it decides the check isn't necessary — that's what Part 2 fixes.

## Part 2 — the mandatory gate

`relayshield_mcp_gate` is a `@tool_input_guardrail`: a hook the SDK runs *before* a tool executes, with the power to block the call outright. Attach it directly to whichever tool actually performs the connect/install — the guardrail is scoped by which tool you assign it to, not by matching tool names inside the gate itself.

The example below stands in for a real `connect_mcp_server` implementation with a stub that just returns a success message, so the notebook is runnable without a live MCP server on the other end. Swap `stub_connect_mcp_server`'s body for your actual connection logic; the gate wraps whatever tool you attach it to.

In [ ]:
from agents import function_tool
from openai_agents_relayshield.guardrail import relayshield_mcp_gate


@function_tool
def stub_connect_mcp_server(api_key: str, server_url: str) -> str:
    """Connect to an MCP server. Stubbed for this example — replace with your real connection logic."""
    return f"Connected to {server_url}"


# The gate runs before stub_connect_mcp_server, using the same server_url
# argument the model passed to the tool call.
stub_connect_mcp_server.tool_input_guardrails = [relayshield_mcp_gate]

gated_agent = Agent(
    name="Assistant",
    instructions="You connect to MCP servers on the user's behalf when asked.",
    tools=[stub_connect_mcp_server],
)

Note that in a real deployment, the raw connect/install tool should never be bound to the model directly without the gate attached — the whole point is that the protected action is unreachable except through the check.

In [ ]:
result = await Runner.run(
    gated_agent,
    f"My RelayShield key is {RELAYSHIELD_API_KEY}. Connect to the MCP server at https://mcp.example.com/sse.",
)
print(result.final_output)

If `check_mcp_server_risk` returns a finding for that server, the gate returns `ToolGuardrailFunctionOutput.reject_content(...)` and `stub_connect_mcp_server` never executes — the model sees a message explaining the block instead of a successful connection. On a clean result, the gate allows the call through and the tool runs normally.

The gate's decision policy, all non-negotiable by design:

- A hook exception defaults to `defer` (blocked, with an explanatory message), never silently to `allow` — a gate failure must not become a pass.
- Bounded retry applies only to transient upstream failures (timeout / 429 / 5xx); auth failures, malformed responses, and payment-required states are terminal after one attempt.
- The gate logs the decision, reason codes, check version, target, and timestamp — never keys, payment proofs, or session material.

## Further reading

- [`openai-agents-relayshield` on GitHub](https://github.com/nzdsf2-gif/openai-agents-relayshield) — source, tests, and the full gate design writeup
- [RelayShield developer docs](https://api.relayshield.net/developers) — API keys and the full endpoint list
- The same normalized gate policy is also available for [LangChain](https://github.com/nzdsf2-gif/langchain-relayshield), built on that framework's `wrap_tool_call` middleware instead of `@tool_input_guardrail` — same decision logic, ported natively to each framework's own hook shape rather than forcing one shape onto both.